# U-Net 二段階学習（Google Colab）— 骨格モード

1. **パッチ化** … 合成 1,000 + Guitar-TECHS（**骨格 → フル** ペア）
2. **Stage 1** … 合成データで学習
3. **Stage 2** … Guitar-TECHS でファインチューニング
4. **push** … `checkpoints/stage2/unet_last.pt` を GitHub へ

**学習タスク（セル 5）:** 既定は `downbeat_chord`（小節頭コード骨格 → リッチなギターパート）。  
旧 `onset_to_full` チェックポイントとは **別タスク** のため、骨格用には **再学習が必要** です。

**事前設定（セル 1）**
- リポジトリが **Private** の場合: `GITHUB_TOKEN`（PAT, repo 権限）が **clone 時から必須**
- リポジトリが **Public** の場合: `GITHUB_TOKEN` は checkpoint push 時のみ必要

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ← Private リポジトリならここに PAT（repo 権限）を入れる

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

# 作業ディレクトリを /content に戻す（再実行時の安全策）
if not Path("train.py").exists() and Path("/content").exists():
    os.chdir("/content")

if Path("train.py").exists():
    print("既にリポジトリ内です:", os.getcwd())
else:
    if REPO_DIR.exists() and not (REPO_DIR / "train.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "train.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、セル1で GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("train.py").exists():
        raise FileNotFoundError(
            "clone 失敗。GITHUB_TOKEN を設定するか、ランタイムを再起動して再実行してください。"
        )

print("OK:", os.getcwd())
!ls train.py prepare_all.py colab_train.ipynb

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
SYNTHETIC_COUNT = 6000
SYNTHETIC_SEED = 42
SYNTHETIC_BARS = 8  # 8固定（長くするとパッチ爆発でディスク落ち）
# downbeat_chord | root_per_bar | melody_line | onset_to_full
PAIR_MODE = "downbeat_chord"

# 容量不足で落ちた場合: ランタイム初期化後、下を実行してから本セルへ
# !rm -rf data/pairs data/raw/synthetic

!python prepare_all.py \
  --synthetic-count {SYNTHETIC_COUNT} \
  --synthetic-seed {SYNTHETIC_SEED} \
  --bars {SYNTHETIC_BARS} \
  --force-regenerate \
  --mode {PAIR_MODE}

In [ ]:
STAGE1_EPOCHS = 40   # 40〜80 で大きな差は見込みにくいので 40 で十分
BATCH_SIZE = 16
POS_WEIGHT = 20      # onsetセルは全体の約0.05%。無音に潰れるのを防ぐ強めの重み

# Stage1: 合成データで基礎（進行・リズム）を学習。lr=1e-3 でしっかり収束させる
#   ※ loss が下がらず発散する場合は --lr 5e-4 に下げる
!python train.py \
  --pairs-dir data/pairs/synthetic \
  --epochs {STAGE1_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr 1e-3 \
  --checkpoint-dir checkpoints/stage1 \
  --pos-weight {POS_WEIGHT}

In [ ]:
# Stage2（弱い微調整）: curated TECHS のみ
#   P3_music 全件 + P1/P2_chords 各最大6本。全量 TECHS はリズムを崩すので使わない。
#   lr/epoch は小さく Stage1 の刻みを守る。
STAGE2_EPOCHS = 5
STAGE2_LR = 1e-5

!python train.py \
  --pairs-dir data/pairs/guitar-techs-curated \
  --epochs {STAGE2_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {STAGE2_LR} \
  --checkpoint-dir checkpoints/stage2 \
  --resume checkpoints/stage1/unet_last.pt \
  --pos-weight {POS_WEIGHT}

In [ ]:
import json
from pathlib import Path

manifest_path = Path("data/pairs/manifest_all.json")
pair_mode = json.loads(manifest_path.read_text(encoding="utf-8"))["mode"] if manifest_path.exists() else PAIR_MODE

!git config user.email "colab@users.noreply.github.com"
!git config user.name "Colab Train"
# curated Stage2（弱微調整）後は stage2 を push。リズムが崩れたら stage1 に差し替え
!python scripts/push_checkpoint.py \
  --ckpt checkpoints/stage2/unet_last.pt \
  --message "Stage2 curated TECHS (P3_music+chords薄) {pair_mode}"

## （オプション）リード学習

進行のコードトーン骨格 → **単音リード** を学習する。backing とは別データ・別チェックポイント（`checkpoints/lead`）。
- 入力: backing と共通の骨格（小節頭コードトーン）
- 正解: スケール上の単音リード（`generate_lead_pairs.py` が生成）
- Stage2 は行わない（合成のみ 1 段。backing で Stage2 がリズムを崩した教訓）

In [ ]:
# リード用 input/target ペアを生成（進行のコード骨格 → 単音リード）
LEAD_COUNT = 6000
!python generate_lead_pairs.py \
  --pairs-dir data/pairs/lead \
  --count {LEAD_COUNT} \
  --seed 42

In [ ]:
# リード学習（合成のみ 1 段。backing と同じ設定で収束させる）
!python train.py \
  --pairs-dir data/pairs/lead \
  --epochs 40 \
  --batch-size 16 \
  --lr 1e-3 \
  --checkpoint-dir checkpoints/lead \
  --pos-weight 20

In [ ]:
# リードのチェックポイントを push（ローカルでは checkpoints/lead/ に配置）
!python scripts/push_checkpoint.py \
  --ckpt checkpoints/lead/unet_last.pt \
  --message "lead synthetic (単音リード)"